# Netflix Movies Dataset — Data Cleaning Project
## Complete Solution Notebook

---

**Topic:** Real-World Data Cleaning with NumPy and Pandas  
**Dataset:** `netflix_movies_dirty (1).csv`

---

### What you will learn in this notebook

By the end of this project you will be able to:

1. Load a messy, real-world dataset and understand its shape and structure
2. Identify every type of data quality problem a dataset can have
3. Write Python functions that clean data step by step
4. Apply those functions to a full DataFrame using `.apply()`
5. Detect and handle missing values, duplicates, wrong data types, and outliers
6. Produce a clean, analysis-ready dataset

---

> **How to read this notebook:** Every section follows this pattern:
> - **Problem Found** — what we discovered
> - **Why is this a problem?** — the real-world impact
> - **How can we solve it?** — the plan before we code
> - Code — beginner-friendly solution
> - Explanation — what just happened

Let's begin.

---
# Step 1 — Import Libraries and Load the Dataset

Before we do anything, we import the libraries we will use throughout the project.


In [1]:
import pandas as pd 
import numpy as np 

# open csv file
df=pd.read_csv("netflix_movies_dirty.csv")


In [2]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1560 entries, 0 to 1559
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Movie_ID          1560 non-null   object 
 1   Title             1483 non-null   object 
 2   Genre             1560 non-null   object 
 3   Release_Year      1560 non-null   int64  
 4   Duration          1383 non-null   object 
 5   Rating            1434 non-null   float64
 6   IMDb_Rating       1394 non-null   float64
 7   Votes             1480 non-null   object 
 8   Director          1488 non-null   object 
 9   Country           1560 non-null   object 
 10  Language          1560 non-null   object 
 11  Budget            1438 non-null   object 
 12  Revenue           1405 non-null   object 
 13  Date_Added        1560 non-null   object 
 14  Age_Rating        1560 non-null   object 
 15  Cast              1458 non-null   object 
 16  Production_House  1486 non-null   object 


---
# Step 2 — First Look at the Dataset

A real data analyst never jumps straight into cleaning. First, we look at what we have.


In [3]:
df.shape       
# df.shape[0]    # print no. of rows 
# df.shape[1]    # print no. of cols

(1560, 17)

In [4]:
df.columns

for i in df.columns:
    print(i)

Movie_ID
Title
Genre
Release_Year
Duration
Rating
IMDb_Rating
Votes
Director
Country
Language
Budget
Revenue
Date_Added
Age_Rating
Cast
Production_House


Notice that **every column shows `object` (string) as its dtype** — that is expected, because we loaded everything as text. Our job is to fix the types after we clean the values.

Now let's look for problems systematically.


---
# Step 3 — Finding Missing Values

## **Problem Found**

Some cells in the dataset appear empty. We need to find exactly how many are missing and in which columns.

## **Why is this a problem?**

Missing values cause errors when you try to do calculations or analysis. For example, you cannot compute the average IMDb rating if some values are empty.

## **How can we solve it?**

We use `.isnull().sum()` to count missing values column by column. We also check for **empty strings**, because sometimes a cell looks empty but actually contains `""` — which pandas does not consider as `NaN`.

In [5]:
df.isnull().sum()

Movie_ID              0
Title                77
Genre                 0
Release_Year          0
Duration            177
Rating              126
IMDb_Rating         166
Votes                80
Director             72
Country               0
Language              0
Budget              122
Revenue             155
Date_Added            0
Age_Rating            0
Cast                102
Production_House     74
dtype: int64

We can see that many columns have missing or empty values. We will deal with each one when we reach that column.

For now, let's continue discovering other problems first.


---
# Step 4 — Checking for Duplicate Movie IDs

## **Problem Found**

Each movie should have a **unique ID**. Let's check if any Movie_IDs appear more than once.

## **Why is this a problem?**

If two different rows share the same ID, we cannot tell them apart. Any analysis that relies on Movie_ID (like joining tables) will produce wrong results.

## **How can we solve it?**

We count how many unique IDs we have and compare it to the total number of rows. We also find which IDs appear more than once.

In [6]:
#  total rows vs unique movie_id
duplicate_ids=df['Movie_ID'].nunique()


In [7]:
# find which movie id appears more then once
id_counts=df['Movie_ID'].value_counts()
id_counts[id_counts>1]

Movie_ID
NF6886    4
NF2666    4
NF5819    4
NF7607    3
NF4882    3
         ..
NF7891    2
NF4037    2
NF6774    2
NF5967    2
NF5757    2
Name: count, Length: 213, dtype: int64

## 💻 Solution — Remove Rows with Duplicate Movie IDs

We will keep the **first** occurrence of each Movie_ID and remove the rest.

> **Note:** In a real job, you might investigate each duplicate carefully. Here, we take the safe approach of keeping the first record.


In [8]:
df=df.drop_duplicates('Movie_ID',keep='first')
df['Movie_ID'].nunique()

1316

✅ **Explanation:** `drop_duplicates(subset="Movie_ID", keep="first")` looks at the Movie_ID column only and removes every row that has the same ID as a row that appeared earlier. The first occurrence is kept.


## Handling Rows with Missing Titles

## **Why is this a problem?**

A movie with no title is useless in the dataset — we cannot identify it. We should remove those rows.


✅ **Explanation:** We first clean every title (removing whitespace, newlines, etc.). Any title that becomes empty after cleaning is set to `NaN`. Then we drop those rows entirely because a movie without a name cannot be analysed.


---
# Step 5 — Cleaning the Title Column

## **Problem Found**

The `Title` column has many problems:

1. **Empty titles** — some rows have no title at all
2. **Extra whitespace** — leading spaces, trailing spaces, double spaces
3. **Newline characters** (`\n`) — titles end with invisible line breaks
4. **ALL CAPS titles** — some titles are fully uppercased
5. **Exclamation marks** — titles like `"Inception!!!"`
6. **Excessively long titles** — titles like `"His House (Extended Collector's Edition with Bonus Features and Director's Commentary)"`

## **Why is this a problem?**

If two rows have the title `"Inception"` and `"  inception\n"`, they look like different movies to a computer even though they are the same.

## **How can we solve it?**

We will write a function that handles each problem one by one. Then we apply that function to every title in the dataset.

In [9]:
def clean_title(title):
#     """
#     Task:
#     1. handle missing nAN
#     2. REMOVE /n OR /t
#     3. rempove extra whitespace
#     4. remove punctuation like !!!
#     5. Turncate long titles 
#     6. convert to title case """


    # 1
    if pd.isnull(title): 
        return np.nan

    # 2
    title= title.replace('/n'," ")
    title= title.replace('/t'," ")

    
    
    # 3
    title = " ".join(title.split())  # leading or trailing extra whitespaces are removed 

    # 4 
    title = title.rstrip('!')       # removes all the occurences of the characters passed in the funcn 

    # 5
    if title=='(':
        title=title[:title.index('(')].strip()

    # 6
    title = title.title() 

    return title


df['Title']= df['Title'].apply(clean_title)
df['Title']

0                                            Little Women
1                                         The White Tiger
2       His House (Extended Collector'S Edition With B...
3                                                    Roma
4                             Fear Street Part Three 1666
                              ...                        
1555                                      The Hand Of God
1556                                                 Mass
1557                                                  NaN
1558                             Demon Slayer Mugen Train
1559                                                  NaN
Name: Title, Length: 1316, dtype: object



Now let's write a function to clean each title:



In [10]:
df['Genre']

0              animation
1       Animation, Drama
2                ROMANCE
3        Thriller, Drama
4                Romance
              ...       
1555               Drama
1556            Thriller
1557            THRILLER
1558              ACTION
1559    Animation, Drama
Name: Genre, Length: 1316, dtype: object

In [11]:
def clean_Genre(genre):

    # 1
    if pd.isnull(genre):
        return np.nan

    # 2
    genre=genre.replace('/n',"")
    genre=genre.replace('/t',"")

    # 3
    genre= genre.split(',')[0]

    # 4
    genre=genre.strip()

    # 5
    genre=genre.title()

    return genre

df['Genre']=df['Genre'].apply(clean_Genre)
df['Genre']

0       Animation
1       Animation
2         Romance
3        Thriller
4         Romance
          ...    
1555        Drama
1556     Thriller
1557     Thriller
1558       Action
1559    Animation
Name: Genre, Length: 1316, dtype: object

✅ **Explanation:** The key idea is to pick **one separator** and split on it. We convert all separators (`,`, `|`, `/`) into `|`, then take only the part before the first `|`. This gives us the primary genre. Title Case makes every genre consistent.


---
# Step 6 — Standardising the Genre Column

## **Problem Found**

The `Genre` column has many inconsistencies:

- `"action"`, `"ACTION"`, `"Action"` — same genre, different casing
- `"Action "`, `" Fantasy"`, `"\tDrama"` — leading/trailing spaces and tab characters
- `"Action, Drama"`, `"Action | Drama"`, `"Drama/Comedy"` — multiple genres joined with different separators

## **Why is this a problem?**

If you try to count how many action movies there are, `"action"`, `"ACTION"`, and `"Action "` will all be counted separately. You will get the wrong answer.

## **How can we solve it?**

We will extract only the **first genre** from each row (the primary genre), then standardise its casing and remove extra whitespace. This gives us one clean category per movie.

✅ **Explanation:** We set a sensible boundary: 1900 to 2025. Any value outside that range is treated as an error and replaced with `NaN`. We do not guess the correct year because we have no way of knowing what it should be.


---
# Step 7 — Fixing the Release Year Column

## **Problem Found**

The `Release_Year` column contains **outlier values** that are impossible or unrealistic:

- `1890` — movies did not exist in 1890
- `2099` — that is in the future

## **Why is this a problem?**

If we calculate the average release year, impossible values will pull the result in the wrong direction. We also cannot correctly sort movies by year.

## **How can we solve it?**

We convert the column to numbers. Then we set any year outside a realistic range (1900–2025) to `NaN` (missing), so they are excluded from analysis.

In [12]:
df['Release_Year'].info

lower=(df['Release_Year']<1900).sum()
lower

upper=(df['Release_Year']>2026).sum()
upper

np.int64(45)

In [13]:
def clean_year(year):
    if pd.isnull(year):
        return np.nan
        
    if (year< 1900) or (year>2025):
        return np.nan
        
    return year
df['Release_Year']=df['Release_Year'].apply(clean_year)
print(df['Release_Year'])

0       1996.0
1       2007.0
2       1993.0
3       2004.0
4       2016.0
         ...  
1555    2017.0
1556    1996.0
1557    2016.0
1558    2017.0
1559    2001.0
Name: Release_Year, Length: 1316, dtype: float64


✅ **Explanation:** The key insight is that we handle each format with a separate `if-elif` block. The `"h"` character is our detector for the hours-and-minutes format. For everything else, we extract only the digits. The outlier filter (30–600 minutes) removes impossible values.


---
# Step 9 — Converting the Rating Column

## **Problem Found**

The `Rating` column contains numbers but they are **stored as text (strings)** because we loaded the file with `dtype=str`. Some values are also missing.

## **Why is this a problem?**

You cannot do arithmetic on text. `"8.5" + "7.0"` is `"8.57.0"` in Python, not `15.5`.

## **How can we solve it?**

We use `pd.to_numeric()` to convert the column to floating-point numbers. The `errors="coerce"` argument automatically turns any non-numeric value into `NaN`.

In [14]:
df['Rating'].info
df['Rating']=df['Rating'].astype(float)

✅ **Explanation:** `pd.to_numeric(errors="coerce")` is the cleanest way to convert a text column to numbers when you know some values might not be valid numbers. Any value that cannot become a number silently becomes `NaN`.


---
# Step 10 — Cleaning the IMDb_Rating Column

## **Problem Found**

The IMDb rating scale runs from **0 to 10**. Our dataset contains values outside this range:

- Values **greater than 10** (e.g. `15.0`) — impossible on IMDb
- Values **below 0** (e.g. `-1.0`) — also impossible

## **Why is this a problem?**

These outliers will corrupt any analysis involving IMDb ratings. The maximum, average, and sorting will all be wrong.

## **How can we solve it?**

1. Convert the column to floats
2. Replace any value outside `[0, 10]` with `NaN`

In [15]:
lower=(df['IMDb_Rating']<0).sum()
lower
upper=(df['IMDb_Rating']>10).sum()
upper

np.int64(45)

In [16]:
def clean_imdb(rating):
    if pd.isnull(rating):
        return np.nan
    if (rating<0) or (rating>10):
        return np.nan
    return rating

df['IMDb_Rating']=df['IMDb_Rating'].apply(clean_imdb)
df['IMDb_Rating']

0       9.5
1       4.0
2       3.4
3       7.0
4       5.7
       ... 
1555    5.0
1556    5.8
1557    7.1
1558    NaN
1559    NaN
Name: IMDb_Rating, Length: 1316, dtype: float64

✅ **Explanation:** The IMDb rating range (0–10) is a known domain rule. Any value outside that range is a data entry error, so we set it to `NaN`. This is called **domain-rule validation**.


---
# Step 11(a) — Cleaning duration column

## **Problem Found**



In [21]:
def clean_duration(duration):
    if pd.isnull(duration):
        return np.nan
    if 'h' in duration:
        hours,mins= duration.split('h')
        hours=int(hours)*60
        mins=int(mins.replace('m',''))
        return hours+ mins
    elif "min" in duration:
        duration=duration.replace('mins','')
        duration=duration.replace('min','')
        return int(duration)
        
    else:
        return int(duration)
   

df['Duration']=df['Duration'].apply(clean_duration)
df.Duration

0       193.0
1        87.0
2       117.0
3       160.0
4       113.0
        ...  
1555    135.0
1556    121.0
1557    103.0
1558      NaN
1559    199.0
Name: Duration, Length: 1316, dtype: float64

---
# Step 11(b) — Cleaning the Votes Column

## **Problem Found**

The `Votes` column contains numbers, but some are formatted with **commas** (e.g. `"1,558,685"`). Pandas treats these as text, not numbers.

## **Why is this a problem?**

`"1,558,685"` is a string. You cannot sort or sum strings that look like numbers.

## **How can we solve it?**

We write a function that removes the commas and converts the value to an integer.

In [ ]:
s="1,558,685"
# s.split(',')    # to separate and store in list on the basius of comma 
b="".join( s.split(','))   # .join to join diffrent elements of list
int(b)

In [ ]:
def clean_votes(votes):
    if pd.isnull(votes):
        return np.nan

    votes=(int)("".join(votes.split(',')))

    return votes

df['Votes']=df['Votes'].apply(clean_votes)
df['Votes']

✅ **Explanation:** The `.replace(",", "")` removes the comma separators. Then `.isdigit()` confirms the remaining characters are all digits before we convert to `int`.


---
# Step 12 — Cleaning the Budget Column

## **Problem Found**

The `Budget` column has two problems:

1. **Dollar signs and commas** — e.g. `"$176,890,118"` is text, not a number
2. **Negative values** — e.g. `"-500000"` — a budget cannot be negative

## **Why is this a problem?**

We cannot do any financial analysis (like comparing budget to revenue) until budget is a clean positive number.

## **How can we solve it?**

Remove the `$` and `,`, convert to a number, then set negative values to `NaN`.

In [ ]:
def clean_budget(budget):
  """
  1. Remove $ and ,
  2. Convert to float
  3. set negative values to nan.
  """

  if pd.isnull(budget):
    return np.nan

  budget = budget.replace("$",'').replace(',','')

  budget = float(budget)

  if budget < 0:
    return np.nan

  return budget

df['Budget']=df['Budget'].apply(clean_budget)
df

✅ **Explanation:** We use a `try-except` block because after removing `$` and `,`, there might still be values that cannot be converted (like stray text). The `try-except` catches those safely and returns `NaN`. Then we apply the domain rule: budget ≥ 0.


---
# Step 13 — Cleaning the Revenue Column

## **Problem Found**

The `Revenue` column has:

1. **Commas in numbers** — e.g. `"687,371,362"`
2. **Revenue = 0** — which is unrealistic for a movie that made it onto Netflix

## **Why is this a problem?**

Revenue of 0 is almost certainly a data entry error. A movie that generated no money whatsoever is not meaningful in most analyses.

## **How can we solve it?**

Remove commas, convert to float, then set revenue <= 0 to `NaN`.

In [ ]:
def clean_revenue(revenue):
    """
    1. Remove commas
    2. Convert to float
    3. Set 0 or negative values to NaN
    """

    # Handle missing values
    if pd.isnull(revenue):
        return np.nan

    # Remove commas
    revenue = revenue.replace(',', '')

    # Convert to float
    revenue = float(revenue)

    # Replace 0 or negative values with NaN
    if revenue <= 0:
        return np.nan

    return revenue

df['Revenue']=df['Revenue'].apply(clean_revenue)
df

✅ **Explanation:** The logic is the same as Budget, with one addition: revenue = 0 is also an outlier. We use `<= 0` instead of `< 0` to catch both zero and negative values.


---
# Step 14 — Standardising the Date_Added Column

## **Problem Found**

The `Date_Added` column stores dates in at least **four different formats**:

- `"2024-05-01"` — ISO format (Year-Month-Day)
- `"01/05/2024"` — Day/Month/Year
- `"May 1, 2024"` — Month name Day, Year
- `"20240501"` — Compact format (no separators)

## **Why is this a problem?**

You cannot sort or compare dates stored as text in different formats. `"20230101"` and `"January 1, 2023"` are the same date but Python cannot know that.

## **How can we solve it?**

We use `pd.to_datetime()` with `infer_datetime_format=True`, which is smart enough to recognise most formats automatically. For the compact format (`"20230101"`), we add a special pre-processing step.

In [ ]:
df['Date_Added'] = pd.to_datetime(df['Date_Added'],format='mixed',errors='coerce')
df.Date_Added

✅ **Explanation:** The compact format (`"20240501"`) is the tricky one — pandas cannot detect it automatically. We handle it by checking if the string is 8 digits long, then inserting dashes to convert it to `"2024-05-01"`. After that, `pd.to_datetime()` handles everything else.


---
# Step 15 — Standardising the Country Column

## **Problem Found**

The same countries appear with different names:

- `"USA"`, `"United States"`, `"U.S."`, `"US"`, `"U.S.A"` — all mean the United States
- `"UK"`, `"United Kingdom"`, `"Britain"`, `"U.K."` — all mean the United Kingdom

## **Why is this a problem?**

If we group movies by country, the USA will appear as 5 separate groups instead of one. We will grossly undercount American movies.

## **How can we solve it?**

We build a **dictionary** that maps every known variant to the official name. Then we write a function that looks up each value in the dictionary.

In [ ]:
d={
    'U.S.':'United States',
    'U.S.A.':'United States',
    'US':'United States',
    'USA':'United States',
    'Britain': 'United Kingdom',
    'U.K.':'United Kingdom',
    'UK':'United Kingdom',
    
}

In [ ]:
df['Country'].unique()

In [ ]:
df.replace(['U.S.','U.S.A.'],'United States')


def clean_country(value):
    
    if pd.isnull(value):
        return np.nan

    if value in d:
        return d[value]


    return value

df['Country']=df['Country'].apply(clean_country)
df
        

✅ **Explanation:** A dictionary is the perfect tool for this problem. We store every known variant as a key (all lowercase) and the correct name as the value. Converting the input to lowercase before lookup means we handle `"USA"`, `"usa"`, and `"Usa"` all the same way.


---
# Step 16 — Standardising the Language Column

## **Problem Found**

The `Language` column has:

- Inconsistent casing: `"english"`, `"ENGLISH"`, `"English"`
- Abbreviations: `"Eng"`, `"Hin"`, `"Kor"`, `"Ger"`, `"Fre"`, `"Jap"`, `"Spa"`, `"Ita"`

## **Why is this a problem?**

`"english"` and `"ENGLISH"` will be counted as different languages. Abbreviations like `"Eng"` might not even be recognised in reports.

## **How can we solve it?**

Build a dictionary mapping abbreviations and incorrect casings to the full, properly-capitalised language name.


In [ ]:
df['Language'].unique()


lang = {
    'english': 'English',
    'English': 'English',
    'Eng' : 'English',
    'Hin' : 'Hindi',
    'Kor' : 'Korean',
    'Ger' : 'German',
    'Fre' : 'French',
    'Jap' : 'Japanese',
    'Spa' : 'Spanish',
    'Ita' : 'Italian',
    'Chinese' : 'Chinese'
}

In [ ]:
def clen_lang(value):
    if pd.isnull(value):
        return np.nan
    if value in lang:
        return lang[value]


    return value.title()

In [ ]:
df['Language']=df['Language'].apply(clen_lang)
df.Language

✅ **Explanation:** The same dictionary-lookup pattern we used for Country works perfectly here. All abbreviations like `"Hin"` become `"Hindi"` and all casing variants like `"ENGLISH"` become `"English"`.


✅ **Explanation:** We reuse the same simple function for all three columns. It strips whitespace and converts empty strings to `NaN`, so all missing values are represented the same way.


---
# Step 17 — Final Duplicate Check (Near-Duplicate Rows)

## **Problem Found**

Even after removing duplicate Movie_IDs, the dataset may still contain rows that are very similar — the same title and year but with slightly different other values.

## **Why is this a problem?**

If the same movie appears twice with slightly different data, it will be double-counted in analysis.

## **How can we solve it?**

We check for rows where both `Title` AND `Release_Year` are the same. For any such group, we keep only the first row.

In [ ]:
df.duplicated(subset=['Title','Release_Year']).sum()

In [ ]:
df.drop_duplicates(subset=["Title", "Release_Year"],keep='first',inplace=True)
df.duplicated(subset=['Title','Release_Year']).sum()

✅ **Explanation:** `drop_duplicates(subset=["Title", "Release_Year"])` looks at both columns together. Two rows must share both the same title AND the same year to be considered duplicates. This avoids accidentally removing movies that share a title but are from different years (like sequels or remakes).


---
# Step 19 — Filling Remaining Missing Values

## **Problem Found**

After all the cleaning above, some columns still have missing values. We need a strategy for each one.

## Strategy by column

| Column | Strategy | Reason |
|---|---|---|
| `Duration` | Fill with median | Median is not affected by outliers |
| `Rating` | Fill with median | Same reason |
| `IMDb_Rating` | Fill with median | Same reason |
| `Votes` | Fill with median | Same reason |
| `Budget` | Leave as NaN | Cannot guess budget — too important to fake |
| `Revenue` | Leave as NaN | Same |
| `Release_Year` | Leave as NaN | Wrong year is worse than no year |
| `Director`, `Cast`, `Production_House` | Leave as NaN | Cannot guess names |

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
num_cols=df.select_dtypes(['int64','float64']).skew()
num_cols

In [ ]:
# skewness-- median
# no near to zero skewness--- mean 


# skewness 
#  0- centralized data
# >0- data is right shifted or right skewed
#   <0 - data is left shifted or left skewed 

df['Release_Year']=df['Release_Year'].fillna(df['Release_Year'].median()) 

df['IMDb_Rating']=df['IMDb_Rating'].fillna(df['IMDb_Rating'].median())   

df['Title']=df['Title'].fillna('Unknown')



✅ **Explanation:** We use `np.median()` to compute the median from a NumPy array. The median is better than the mean for filling missing values because it is not pulled by extreme values. We intentionally leave Budget, Revenue, and year-related columns as `NaN` because guessing wrong values would be worse than having no value.


---
# Step 20 — Final Data Type Conversion

## **Problem Found**

Some columns that should be integers (like `Release_Year`, `Votes`, `Duration`) are still stored as floats because of the `NaN` values we introduced during cleaning. Pandas requires integer columns to have no missing values.

## **How can we solve it?**

We use pandas **nullable integer type** (`"Int64"`) which allows integers with missing values. For columns with no missing values, we convert to standard `int`.

✅ **Explanation:** `"Int64"` (capital I) is pandas' nullable integer type. It can store whole numbers AND `NaN` at the same time, which regular Python `int` cannot do.


---
# Step 21 — Final Verification

Now that all cleaning is done, let's verify that our dataset is truly clean.


---
# Step 22 — Before vs After Comparison

Let's compare key statistics from the raw dataset to the cleaned dataset.


---
# Step 23 — Save the Cleaned Dataset

Finally, we save the cleaned dataset as a new CSV file so the original messy file is preserved.


---

# 🎉 Project Complete!

Here is a summary of everything we cleaned in this project:

| Step | Problem | Solution |
|------|---------|----------|
| 4 | Duplicate Movie_IDs | `drop_duplicates(subset="Movie_ID")` |
| 5 | Messy titles (spaces, newlines, caps, long) | Custom `clean_title()` function |
| 6 | Inconsistent genre formats | Custom `clean_genre()` + dictionary |
| 7 | Release year outliers (1890, 2099) | Domain-rule validation |
| 8 | Duration in mixed formats | Custom `convert_duration()` function |
| 9 | Rating stored as string | `pd.to_numeric()` |
| 10 | IMDb ratings outside 0–10 | Domain-rule validation |
| 11 | Votes with commas | Remove commas, convert to int |
| 12 | Budget with `$` signs and negatives | Remove `$`/`,`, check ≥ 0 |
| 13 | Revenue with commas, zero values | Remove `,`, check > 0 |
| 14 | Dates in 4 different formats | `pd.to_datetime()` + compact format handler |
| 15 | Country name variants | Lookup dictionary |
| 16 | Language abbreviations and casing | Lookup dictionary |
| 17 | Missing Director/Cast/Production House | Standardised to `NaN` |
| 18 | Near-duplicate rows (same Title + Year) | `drop_duplicates(subset=["Title","Release_Year"])` |
| 19 | Remaining missing numeric values | Fill with median |
| 20 | Wrong data types | `astype()` and `"Int64"` |

**Key Python concepts used:**

- `def` functions with clear parameters
- `for` loops and `if-elif-else` conditions
- `str.strip()`, `str.replace()`, `str.split()`, `str.lower()`, `str.title()`
- Dictionaries for lookup tables
- NumPy arrays and aggregate functions
- `pd.to_numeric()`, `pd.to_datetime()`
- `df.apply()` to run a function on every row
- `df.dropna()`, `df.fillna()`, `df.drop_duplicates()`
